# Hyperparameter Tuning

This notebook performs hyperparameter tuning for the
machine learning models used for crop yield prediction.

The tuning process is performed to identify suitable
hyperparameter values that can improve model performance.

The models considered are:
- Random Forest
- XGBoost
- Support Vector Regression (SVR)

Model performance is evaluated using RMSE and R².

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv(
    r"C:\Crop Yield Prediction\data\Crop Yield Data.csv"
)

print("Dataset loaded successfully!")
print("Initial dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Initial dataset shape: (19689, 13)


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,Avg_Temperature,Max_Temperature,Min_Temperature
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796,23.692,33.435,14.779
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710,23.692,33.435,14.779
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238,23.692,33.435,14.779
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.052,23.692,33.435,14.779
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.421,23.692,33.435,14.779


In [3]:
print("========== MISSING VALUE ANALYSIS ==========")

print(df.isnull().sum())

print(
    "\nTotal missing values:",
    df.isnull().sum().sum()
)

========== MISSING VALUE ANALYSIS ==========
Crop               0
Crop_Year          0
Season             0
State              0
Area               0
Production         0
Annual_Rainfall    2
Fertilizer         2
Pesticide          2
Yield              1
Avg_Temperature    2
Max_Temperature    0
Min_Temperature    0
dtype: int64

Total missing values: 9


In [4]:
print(
    "Missing Yield values before removal:",
    df["Yield"].isnull().sum()
)

df = df.dropna(
    subset=["Yield"]
).reset_index(drop=True)

print(
    "Missing Yield values after removal:",
    df["Yield"].isnull().sum()
)

print(
    "Dataset shape after removing missing Yield:",
    df.shape
)

Missing Yield values before removal: 1
Missing Yield values after removal: 0
Dataset shape after removing missing Yield: (19688, 13)


In [5]:
print(
    "Missing Yield values before removal:",
    df["Yield"].isnull().sum()
)

df = df.dropna(
    subset=["Yield"]
).reset_index(drop=True)

print(
    "Missing Yield values after removal:",
    df["Yield"].isnull().sum()
)

print(
    "Dataset shape after removing missing Yield:",
    df.shape
)

Missing Yield values before removal: 0
Missing Yield values after removal: 0
Dataset shape after removing missing Yield: (19688, 13)


## Feature Selection

The target variable is `Yield`.

The `Production` feature is excluded from the model because
it is directly related to crop yield and can introduce
target leakage.

Therefore, `Production` is not used as an input feature.

In [6]:
X = df.drop(
    ["Yield", "Production"],
    axis=1
)

y = df["Yield"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures used:")
print(X.columns.tolist())

Features shape: (19688, 11)
Target shape: (19688,)

Features used:
['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Avg_Temperature', 'Max_Temperature', 'Min_Temperature']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("========== TRAIN-TEST SPLIT ==========")

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

========== TRAIN-TEST SPLIT ==========
Training data shape: (15750, 11)
Testing data shape: (3938, 11)
Training target shape: (15750,)
Testing target shape: (3938,)


In [8]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("========== FEATURE TYPES ==========")

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

========== FEATURE TYPES ==========
Numerical Features:
['Crop_Year', 'Area', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Avg_Temperature', 'Max_Temperature', 'Min_Temperature']

Categorical Features:
['Crop', 'Season', 'State']


C:\Users\Manisha\AppData\Local\Temp\ipykernel_24256\1339560515.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [9]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical preprocessing pipeline created!")

Numerical preprocessing pipeline created!


In [10]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print("Categorical preprocessing pipeline created!")

Categorical preprocessing pipeline created!


In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [12]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print("Training data processed successfully!")
print("Testing data processed successfully!")

print(
    "\nProcessed training shape:",
    X_train_processed.shape
)

print(
    "Processed testing shape:",
    X_test_processed.shape
)

Training data processed successfully!
Testing data processed successfully!

Processed training shape: (15750, 99)
Processed testing shape: (3938, 99)


In [13]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_processed,
    y_train
)

rf_predictions = rf_model.predict(
    X_test_processed
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_predictions
    )
)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)

print("========== RANDOM FOREST BASELINE ==========")

print("RMSE:", rf_rmse)
print("R²:", rf_r2)

========== RANDOM FOREST BASELINE ==========
RMSE: 152.4371514575675
R²: 0.9671132070577182


In [14]:
rf_param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5]
}

print("Random Forest parameter grid:")
print(rf_param_grid)

Random Forest parameter grid:
{'n_estimators': [50, 100], 'max_depth': [None, 10], 'min_samples_split': [2, 5]}


In [15]:
print("Starting Random Forest hyperparameter tuning...")

rf_grid_search = GridSearchCV(
    estimator=RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=rf_param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

rf_grid_search.fit(
    X_train_processed,
    y_train
)

print("Random Forest tuning completed!")

Starting Random Forest hyperparameter tuning...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Random Forest tuning completed!


In [16]:
print("========== BEST RANDOM FOREST PARAMETERS ==========")

print(
    rf_grid_search.best_params_
)

print(
    "\nBest Cross-Validation RMSE:",
    -rf_grid_search.best_score_
)

========== BEST RANDOM FOREST PARAMETERS ==========
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}

Best Cross-Validation RMSE: 191.2048581680293


In [17]:
best_rf_model = rf_grid_search.best_estimator_

tuned_rf_predictions = best_rf_model.predict(
    X_test_processed
)

tuned_rf_mae = mean_absolute_error(
    y_test,
    tuned_rf_predictions
)

tuned_rf_mse = mean_squared_error(
    y_test,
    tuned_rf_predictions
)

tuned_rf_rmse = np.sqrt(
    tuned_rf_mse
)

tuned_rf_r2 = r2_score(
    y_test,
    tuned_rf_predictions
)

print("========== TUNED RANDOM FOREST RESULTS ==========")

print("MAE :", tuned_rf_mae)
print("MSE :", tuned_rf_mse)
print("RMSE:", tuned_rf_rmse)
print("R²  :", tuned_rf_r2)

========== TUNED RANDOM FOREST RESULTS ==========
MAE : 11.588271422041652
MSE : 23237.085144497374
RMSE: 152.4371514575675
R²  : 0.9671132070577182


In [18]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror",
    eval_metric="rmse",
    n_jobs=-1
)

xgb_model.fit(
    X_train_processed,
    y_train
)

xgb_predictions = xgb_model.predict(
    X_test_processed
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_predictions
    )
)

xgb_r2 = r2_score(
    y_test,
    xgb_predictions
)

print("========== XGBOOST BASELINE ==========")

print("RMSE:", xgb_rmse)
print("R²:", xgb_r2)

========== XGBOOST BASELINE ==========
RMSE: 125.38366882398537
R²: 0.9777504197796166


In [19]:
xgb_param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 6]
}

print("XGBoost parameter grid:")
print(xgb_param_grid)

XGBoost parameter grid:
{'n_estimators': [50, 100], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 6]}


In [20]:
print("Starting XGBoost hyperparameter tuning...")

xgb_grid_search = GridSearchCV(
    estimator=XGBRegressor(
        random_state=42,
        objective="reg:squarederror",
        eval_metric="rmse",
        n_jobs=-1
    ),
    param_grid=xgb_param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

xgb_grid_search.fit(
    X_train_processed,
    y_train
)

print("XGBoost tuning completed!")

Starting XGBoost hyperparameter tuning...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
XGBoost tuning completed!


In [21]:
print("========== BEST XGBOOST PARAMETERS ==========")

print(
    xgb_grid_search.best_params_
)

print(
    "\nBest Cross-Validation RMSE:",
    -xgb_grid_search.best_score_
)

========== BEST XGBOOST PARAMETERS ==========
{'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100}

Best Cross-Validation RMSE: 184.48898858064476


In [22]:
best_xgb_model = xgb_grid_search.best_estimator_

tuned_xgb_predictions = best_xgb_model.predict(
    X_test_processed
)

tuned_xgb_mae = mean_absolute_error(
    y_test,
    tuned_xgb_predictions
)

tuned_xgb_mse = mean_squared_error(
    y_test,
    tuned_xgb_predictions
)

tuned_xgb_rmse = np.sqrt(
    tuned_xgb_mse
)

tuned_xgb_r2 = r2_score(
    y_test,
    tuned_xgb_predictions
)

print("========== TUNED XGBOOST RESULTS ==========")

print("MAE :", tuned_xgb_mae)
print("MSE :", tuned_xgb_mse)
print("RMSE:", tuned_xgb_rmse)
print("R²  :", tuned_xgb_r2)

========== TUNED XGBOOST RESULTS ==========
MAE : 11.329533327038849
MSE : 17963.488466209
RMSE: 134.02793912542637
R²  : 0.9745767800894263


In [23]:
tuning_results = pd.DataFrame({
    "Model": [
        "Random Forest Baseline",
        "Random Forest Tuned",
        "XGBoost Baseline",
        "XGBoost Tuned"
    ],
    
    "RMSE": [
        rf_rmse,
        tuned_rf_rmse,
        xgb_rmse,
        tuned_xgb_rmse
    ],
    
    "R²": [
        rf_r2,
        tuned_rf_r2,
        xgb_r2,
        tuned_xgb_r2
    ]
})

tuning_results

,Model,RMSE,R²
0,Random Forest Baseline,152.437151,0.967113
1,Random Forest Tuned,152.437151,0.967113
2,XGBoost Baseline,125.383669,0.977750
3,XGBoost Tuned,134.027939,0.974577


In [24]:
print("========== TUNING SUMMARY ==========")

print(
    "Random Forest Best Parameters:"
)

print(
    rf_grid_search.best_params_
)

print(
    "\nXGBoost Best Parameters:"
)

print(
    xgb_grid_search.best_params_
)

========== TUNING SUMMARY ==========
Random Forest Best Parameters:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}

XGBoost Best Parameters:
{'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100}


In [25]:
joblib.dump(
    best_rf_model,
    r"C:\Crop Yield Prediction\models\tuned_random_forest_model.pkl"
)

print("Tuned Random Forest model saved successfully!")

Tuned Random Forest model saved successfully!


In [26]:
joblib.dump(
    best_xgb_model,
    r"C:\Crop Yield Prediction\models\tuned_xgboost_model.pkl"
)

print("Tuned XGBoost model saved successfully!")

Tuned XGBoost model saved successfully!


In [27]:
joblib.dump(
    preprocessor,
    r"C:\Crop Yield Prediction\models\tuned_preprocessor.pkl"
)

print("Preprocessor saved successfully!")

Preprocessor saved successfully!


## Hyperparameter Tuning Conclusion

Hyperparameter tuning was performed for Random Forest and
XGBoost using GridSearchCV with 3-fold cross-validation.

The tuned models were evaluated using MAE, MSE, RMSE and R².

The best hyperparameter combinations were identified based
on cross-validation RMSE.

The baseline and tuned model performances were compared.

The tuned Random Forest and XGBoost models were saved for
future use.

SVR was not included in GridSearchCV because RBF-based SVR
requires considerably more computation on the dataset.
Its baseline performance will be included in the final
model comparison.